In [2]:
# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)

SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True


## 13) GCN (Graph Convolutional Networks) – Ứng dụng: **Phân loại node bán giám sát** (Karate Club)

**Công thức Kipf & Welling (2017):**  
\(
\hat{A} = A + I,\quad \hat{D}_{ii} = \sum_j \hat{A}_{ij},\quad \tilde{A} = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}
\)  
\(
H^{(1)} = \mathrm{ReLU}(\tilde{A} X W^{(0)}),\quad
Z = \tilde{A} H^{(1)} W^{(1)}
\)

Ta dùng cùng đồ thị Karate, train với 10 node gán nhãn.

In [4]:
import numpy as np

if not 'NETWORKX_OK' in globals() or not NETWORKX_OK:
    print("Cần networkx cho ví dụ này.")
else:
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G)
    I = np.eye(n)
    A_hat = A + I
    D_hat = np.diag( A_hat.sum(axis=1) )
    D_inv_sqrt = np.linalg.inv(np.sqrt(D_hat + 1e-8*np.eye(n)))
    A_tilde = D_inv_sqrt @ A_hat @ D_inv_sqrt    # normalized adjacency

    # labels
    labels = np.array([0 if G.nodes[i]['club']=='Mr. Hi' else 1 for i in range(n)])

    rng = np.random.default_rng(123)
    X = rng.normal(0,1,size=(n, 16)).astype(np.float32)

    idx_all = np.arange(n)
    rng.shuffle(idx_all)
    idx_train = idx_all[:10]
    idx_test  = idx_all[10:]

    d_in, d_h, d_out = X.shape[1], 32, 2
    W0 = rng.normal(0, 0.1, size=(d_in, d_h))
    W1 = rng.normal(0, 0.1, size=(d_h, d_out))

    def relu(z): return np.maximum(z,0)
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True); e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def forward(X):
        H1 = relu(A_tilde @ X @ W0)   # GCN layer 1
        Z  = A_tilde @ H1 @ W1        # GCN layer 2
        P  = softmax(Z)
        return H1, P

    def loss_and_grads(X, labels, idx):
        H1, P = forward(X)
        Y = np.zeros((n, d_out)); Y[np.arange(n), labels] = 1.0

        mask = np.zeros((n,1)); mask[idx] = 1.0
        L = - (mask * (Y * np.log(P+1e-9)).sum(axis=1, keepdims=True)).sum() / mask.sum()

        # Backprop
        dZ = (P - Y) * mask  # [n, d_out]
        dW1 = (H1.T @ (A_tilde.T @ dZ))
        dH1 = (A_tilde.T @ dZ) @ W1.T
        dH1[H1<=0] = 0
        dW0 = (X.T @ (A_tilde.T @ dH1))

        return float(L), dW0, dW1

    lr = 0.1
    for ep in range(1, 401):
        L, dW0, dW1 = loss_and_grads(X, labels, idx_train)
        W0 -= lr * dW0; W1 -= lr * dW1
        if ep % 50 == 0:
            _, P = forward(X)
            pred = P.argmax(axis=1)
            acc_train = (pred[idx_train] == labels[idx_train]).mean()
            acc_test  = (pred[idx_test]  == labels[idx_test]).mean()
            print(f"Epoch {ep:3d} | loss={L:.3f} | acc_train={acc_train:.2f} | acc_test={acc_test:.2f}")

Epoch  50 | loss=0.019 | acc_train=1.00 | acc_test=0.88
Epoch 100 | loss=0.007 | acc_train=1.00 | acc_test=0.88
Epoch 150 | loss=0.004 | acc_train=1.00 | acc_test=0.88
Epoch 200 | loss=0.003 | acc_train=1.00 | acc_test=0.88
Epoch 250 | loss=0.002 | acc_train=1.00 | acc_test=0.88
Epoch 300 | loss=0.001 | acc_train=1.00 | acc_test=0.88
Epoch 350 | loss=0.001 | acc_train=1.00 | acc_test=0.88
Epoch 400 | loss=0.001 | acc_train=1.00 | acc_test=0.88
